In [24]:
!pip install optuna

In [52]:
import os
import numpy as np
import pandas as pd

In [53]:
import gdown
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error,mean_squared_error,mean_absolute_percentage_error as mape
from lightgbm import LGBMRegressor

tscv = TimeSeriesSplit(n_splits=3)

In [54]:
import warnings
warnings.filterwarnings('ignore')

In [55]:
datafile_dict={
    "train_transformed_data.csv": "1w_ZOLTTc8WnLj7l8bnwo8Vuz1ii_ZBTY"
    ,"validate_transformed_data.csv" : "1SI5pS8igjgATw5fSPB8hYt5vc62aiVHF"
    ,"test_data.csv" : "1JhzqIAupBjD-vZTqnaAqL1OI9HIpD9O6"
}
#remove existing files
for filename in datafile_dict.keys():
    if os.path.exists(filename):
        os.remove(filename)
        print(f"Removed {filename}")
    else:
        print(f"{filename} not found, skipping removal.")
#doanload new files from drive
for filename, file_id in datafile_dict.items():
  gdown.download(id=file_id, output=filename, quiet=False)

Removed train_transformed_data.csv
Removed validate_transformed_data.csv
Removed test_data.csv


Downloading...
From: https://drive.google.com/uc?id=1w_ZOLTTc8WnLj7l8bnwo8Vuz1ii_ZBTY
To: /content/train_transformed_data.csv
100%|██████████| 28.5M/28.5M [00:00<00:00, 204MB/s]
Downloading...
From: https://drive.google.com/uc?id=1SI5pS8igjgATw5fSPB8hYt5vc62aiVHF
To: /content/validate_transformed_data.csv
100%|██████████| 6.29M/6.29M [00:00<00:00, 229MB/s]
Downloading...
From: https://drive.google.com/uc?id=1JhzqIAupBjD-vZTqnaAqL1OI9HIpD9O6
To: /content/test_data.csv
100%|██████████| 572k/572k [00:00<00:00, 124MB/s]


In [56]:
def load_data_dat(filename):
  if filename.endswith('.csv'):
    df=pd.read_csv(filename)
  return df

train_df= load_data_dat('train_transformed_data.csv')
val_df= load_data_dat('validate_transformed_data.csv')
test_df= load_data_dat('test_data.csv')

In [57]:
FEATURES = [
    'Store_id',#'Store_Type','Location_Type','Region_Code',
    'Store_Type_S1',
       'Store_Type_S2', 'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4',
    'Discount_Flag','Holiday',
    'day','month','weekday','weekofyear','is_weekend',
    'lag_1_#Order','lag_7_#Order','rolling_#Order_7',
    'lag_1_Sales','lag_7_Sales','rolling_Sales_7',
    'lag_1_aov',
    'Sales','#Order'
]
FEATURES_ORDERS = [
    'Store_id',#'Store_Type','Location_Type','Region_Code',
    'Store_Type_S1',
       'Store_Type_S2', 'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4',
    'Discount_Flag','Holiday',
    'day','month','weekday','weekofyear','is_weekend',
    'lag_1_#Order','lag_7_#Order','rolling_#Order_7',
    'lag_1_Sales','lag_7_Sales','rolling_Sales_7',
    'lag_1_aov'
]

FEATURES_SALES = [
    'Store_id',#'Store_Type','Location_Type','Region_Code',
    'Store_Type_S1',
       'Store_Type_S2', 'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4',
    'Discount_Flag','Holiday',
    'day','month','weekday','weekofyear','is_weekend',
    'lag_1_Sales','lag_7_Sales','rolling_Sales_7',
    'lag_1_#Order','lag_7_#Order','rolling_#Order_7',
    'lag_1_aov','pred_orders'
]

column_dtype_map_dict = {'Store_id': 'int64',
 'Store_Type_S1': 'int64',
 'Store_Type_S2': 'int64',
 'Store_Type_S3': 'int64',
 'Store_Type_S4': 'int64',
 'Location_Type_L1': 'int64',
 'Location_Type_L2': 'int64',
 'Location_Type_L3': 'int64',
 'Location_Type_L4': 'int64',
 'Location_Type_L5': 'int64',
 'Region_Code_R1': 'int64',
 'Region_Code_R2': 'int64',
 'Region_Code_R3': 'int64',
 'Region_Code_R4': 'int64',
 'Discount_Flag': 'int64',
 'Holiday': 'int64',
 'day': 'int64',
 'month': 'int64',
 'weekday': 'int64',
 'weekofyear': 'int64',
 'is_weekend': 'int64',
 'lag_1_#Order': 'float64',
 'lag_7_#Order': 'float64',
 'rolling_#Order_7': 'float64',
 'lag_1_Sales': 'float64',
 'lag_7_Sales': 'float64',
 'rolling_Sales_7': 'float64',
 'lag_1_aov': 'float64',
 'Sales': 'float64',
 '#Order': 'float64',
 'pred_orders': 'float64'}

In [58]:
train_df['Date'] = pd.to_datetime(train_df['Date'])
val_df['Date'] = pd.to_datetime(val_df['Date'])
train_df = train_df.sort_values("Date")
val_df = val_df.sort_values("Date")
train_df=train_df[FEATURES]
val_df=val_df[FEATURES]

In [59]:
def format_df_dtype(df, features_dtype_map_dict):
    for col, dtype in features_dtype_map_dict.items():
        if col in df.columns:
            df[col] = df[col].astype(dtype)
    return df

train_df = format_df_dtype(train_df, column_dtype_map_dict)
val_df = format_df_dtype(val_df, column_dtype_map_dict)

#**LightGBM**
---

##**1. Orders**
---

In [60]:
TARGET = "#Order"  # or "#Order"

X = train_df[FEATURES_ORDERS]
y = train_df[TARGET]


In [92]:
def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(y_true) + 1e-8)

In [62]:
tscv = TimeSeriesSplit(n_splits=3)
mean_metric_scores = []
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42
    }

    wape_scores = []
    mae_scores = []
    mse_scores = []
    rmse_scores = []
    mape_scores = []

    for train_idx, val_idx in tscv.split(X):

        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        model = LGBMRegressor(**params)
        model.fit(X_train_fold, y_train_fold)

        preds = model.predict(X_val_fold)
        #score = mean_absolute_error(y_val_fold, preds)
        #score = wape(y_val_fold, preds)

        wape_scores.append(wape(y_val_fold, preds))
        mae_scores.append(mean_absolute_error(y_val_fold, preds))
        mse_scores.append(mean_squared_error(y_val_fold, preds))
        rmse_scores.append(np.sqrt(mean_squared_error(y_val_fold, preds)))
        mape_scores.append(mape(y_val_fold, preds))

    average_metric = np.mean(mae_scores)
    mean_score = mean_metric_scores.append((np.mean(wape_scores),np.mean(mae_scores),np.mean(mse_scores),np.mean(rmse_scores),np.mean(mape_scores),params))
    return average_metric

In [63]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best Params:", study.best_params)
print("Best Score:", study.best_value)

[I 2026-04-20 19:19:13,187] A new study created in memory with name: no-name-974dcb30-30f0-4ef9-b57b-64158aaf167a


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002708 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2050
[LightGBM] [Info] Number of data points in the train set: 36135, number of used features: 28
[LightGBM] [Info] Start training from score 67.524810
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004852 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2132
[LightGBM] [Info] Number of data points in the train set: 72270, number of used features: 28
[LightGBM] [Info] Start training from score 68.211554
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008386 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not eno

[I 2026-04-20 19:19:33,457] Trial 0 finished with value: 11.921673866301097 and parameters: {'n_estimators': 530, 'num_leaves': 136, 'learning_rate': 0.29288270670449307, 'subsample': 0.6146023355348684, 'colsample_bytree': 0.8462571673868868}. Best is trial 0 with value: 11.921673866301097.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002395 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2050
[LightGBM] [Info] Number of data points in the train set: 36135, number of used features: 28
[LightGBM] [Info] Start training from score 67.524810
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005295 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2132
[LightGBM] [Info] Number of data points in the train set: 72270, number of used features: 28
[LightGBM] [Info] Start training from score 68.211554
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008162 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not eno

[I 2026-04-20 19:19:52,324] Trial 1 finished with value: 12.15433749122016 and parameters: {'n_estimators': 761, 'num_leaves': 53, 'learning_rate': 0.2529494875327431, 'subsample': 0.7447712179622444, 'colsample_bytree': 0.8566231239968625}. Best is trial 0 with value: 11.921673866301097.


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002426 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2050
[LightGBM] [Info] Number of data points in the train set: 36135, number of used features: 28
[LightGBM] [Info] Start training from score 67.524810
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004493 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2132
[LightGBM] [Info] Number of data points in the train set: 72270, number of used features: 28
[LightGBM] [Info] Start training from score 68.211554
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007750 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not eno

[I 2026-04-20 19:27:02,593] Trial 28 finished with value: 10.98494886033866 and parameters: {'n_estimators': 401, 'num_leaves': 118, 'learning_rate': 0.09011805267436163, 'subsample': 0.6788271999278775, 'colsample_bytree': 0.6970637527685515}. Best is trial 12 with value: 10.382896474085692.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002560 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2050
[LightGBM] [Info] Number of data points in the train set: 36135, number of used features: 28
[LightGBM] [Info] Start training from score 67.524810
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005000 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2132
[LightGBM] [Info] Number of data points in the train set: 72270, number of used features: 28
[LightGBM] [Info] Start training from score 68.211554
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007738 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not eno

[I 2026-04-20 19:27:31,927] Trial 29 finished with value: 10.582227224109019 and parameters: {'n_estimators': 475, 'num_leaves': 140, 'learning_rate': 0.010515532425466718, 'subsample': 0.6226078234167051, 'colsample_bytree': 0.89295634647334}. Best is trial 12 with value: 10.382896474085692.


Best Params: {'n_estimators': 343, 'num_leaves': 114, 'learning_rate': 0.01108019477729205, 'subsample': 0.6017693529578273, 'colsample_bytree': 0.7481295732862226}
Best Score: 10.382896474085692


In [64]:
formatted_data = []
for wape, mae, mse, rmse, mape, params in mean_metric_scores:
    # Combine metrics and the params dictionary into one flat dict
    row = {
        'WAPE': wape, 'MAE': mae, 'MSE': mse, 
        'RMSE': rmse, 'MAPE': mape
    }
    row.update(params)
    formatted_data.append(row)

df = pd.DataFrame(formatted_data)
df

,WAPE,MAE,MSE,RMSE,MAPE,n_estimators,num_leaves,learning_rate,subsample,colsample_bytree,random_state
0,0.175411,11.921674,274.944617,16.524898,3.997833e+13,530,136,0.292883,0.614602,0.846257,42
1,0.178813,12.154337,286.191204,16.857190,4.095294e+13,761,53,0.252949,0.744771,0.856623,42
2,0.164265,11.164301,246.809541,15.626903,4.402408e+13,573,32,0.057018,0.831815,0.649956,42
3,0.177619,12.066519,282.487720,16.778893,4.458311e+13,636,50,0.282459,0.827445,0.791430,42
4,0.169846,11.540432,265.779625,16.245712,4.234197e+13,685,87,0.226315,0.771780,0.720964,42
5,0.162425,11.040073,240.140744,15.424158,4.161602e+13,495,92,0.053849,0.663127,0.739355,42
6,0.164430,11.174408,249.171826,15.723206,4.312601e+13,264,134,0.074333,0.694039,0.911531,42
7,0.170529,11.588568,264.660116,16.222575,4.296025e+13,523,96,0.210664,0.999526,0.626738,42
8,0.165313,11.234897,250.336312,15.762134,4.294972e+13,589,111,0.125053,0.921381,0.962766,42
9,0.177378,12.055928,277.647556,16.597750,4.333154e+13,428,80,0.238308,0.973918,0.667776,42


In [65]:
df.to_csv('lightgbm_errorfile_orders.csv', header=True, index=False)

In [66]:
study.best_params

{'n_estimators': 343,
 'num_leaves': 114,
 'learning_rate': 0.01108019477729205,
 'subsample': 0.6017693529578273,
 'colsample_bytree': 0.7481295732862226}

##**2. Sales**
---

In [93]:
train_df= load_data_dat('train_transformed_data.csv')
val_df= load_data_dat('validate_transformed_data.csv')
test_df= load_data_dat('test_data.csv')

In [94]:
train_df['Date'] = pd.to_datetime(train_df['Date'])
val_df['Date'] = pd.to_datetime(val_df['Date'])
train_df = train_df.sort_values("Date")
val_df = val_df.sort_values("Date")
train_df=train_df[FEATURES]
val_df=val_df[FEATURES]

In [95]:
def format_df_dtype(df, features_dtype_map_dict):
    for col, dtype in features_dtype_map_dict.items():
        if col in df.columns:
            df[col] = df[col].astype(dtype)
    return df

train_df = format_df_dtype(train_df, column_dtype_map_dict)
val_df = format_df_dtype(val_df, column_dtype_map_dict)

In [96]:
TARGET = "#Order"  # or "#Order"

X = train_df[FEATURES_ORDERS]
y = train_df[TARGET]


In [97]:
order_params = {
            'n_estimators': 343,
            'num_leaves': 114,
            'learning_rate': 0.01108019477729205,
            'subsample': 0.6017693529578273,
            'colsample_bytree': 0.7481295732862226
            }
model_orders = LGBMRegressor(**order_params)

model_orders.fit(X, y)
X['pred_orders'] = model_orders.predict(X)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009272 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2177
[LightGBM] [Info] Number of data points in the train set: 144540, number of used features: 28
[LightGBM] [Info] Start training from score 67.815373


In [98]:
X[X['Store_id'] == 1].head(10)

,Store_id,Store_Type_S1,Store_Type_S2,Store_Type_S3,Store_Type_S4,Location_Type_L1,Location_Type_L2,Location_Type_L3,Location_Type_L4,Location_Type_L5,...,weekofyear,is_weekend,lag_1_#Order,lag_7_#Order,rolling_#Order_7,lag_1_Sales,lag_7_Sales,rolling_Sales_7,lag_1_aov,pred_orders
0,1,1,0,0,0,0,0,1,0,0,...,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.459403
1,1,1,0,0,0,0,0,1,0,0,...,1,0,9.0,NaN,NaN,7011.84,NaN,NaN,701.184000,56.238974
2,1,1,0,0,0,0,0,1,0,0,...,1,0,60.0,NaN,NaN,42369.00,NaN,NaN,694.573770,60.774135
3,1,1,0,0,0,0,0,1,0,0,...,1,0,72.0,NaN,NaN,50037.00,NaN,NaN,685.438356,68.634013
4,1,1,0,0,0,0,0,1,0,0,...,1,0,64.0,NaN,NaN,44397.00,NaN,NaN,683.030769,65.637383
5,1,1,0,0,0,0,0,1,0,0,...,1,1,63.0,NaN,NaN,47604.00,NaN,NaN,743.812500,50.714325
6,1,1,0,0,0,0,0,1,0,0,...,1,1,36.0,NaN,NaN,24495.00,NaN,NaN,662.027027,54.544784
7,1,1,0,0,0,0,0,1,0,0,...,2,0,55.0,9.0,51.285714,36855.00,7011.84,36109.834286,658.125000,47.728801
8,1,1,0,0,0,0,0,1,0,0,...,2,0,52.0,60.0,57.428571,34101.00,42369.00,39979.714286,643.415094,50.145698
9,1,1,0,0,0,0,0,1,0,0,...,2,0,65.0,72.0,58.142857,42429.00,50037.00,39988.285714,642.863636,48.580716


In [99]:
TARGET = 'Sales'
y = train_df[TARGET]

In [100]:
tscv = TimeSeriesSplit(n_splits=3)
mean_metric_scores = []
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42
    }

    wape_scores = []
    mae_scores = []
    mse_scores = []
    rmse_scores = []
    mape_scores = []

    for train_idx, val_idx in tscv.split(X):

        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        model = LGBMRegressor(**params)
        model.fit(X_train_fold, y_train_fold)

        preds = model.predict(X_val_fold)
        #score = mean_absolute_error(y_val_fold, preds)
        #score = wape(y_val_fold, preds)

        wape_scores.append(wape(y_val_fold, preds))
        mae_scores.append(mean_absolute_error(y_val_fold, preds))
        mse_scores.append(mean_squared_error(y_val_fold, preds))
        rmse_scores.append(np.sqrt(mean_squared_error(y_val_fold, preds)))
        #mape_scores.append(mape(y_val_fold, preds))

    average_metric = np.mean(mae_scores)
    mean_score = mean_metric_scores.append((
                                np.mean(wape_scores), 
                                np.mean(mae_scores), 
                                np.mean(mse_scores), 
                                np.mean(rmse_scores), 
                                0,  # Placeholder for MAPE since it's commented out
                                params
                            ))
    return average_metric

In [101]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best Params:", study.best_params)
print("Best Score:", study.best_value)

[I 2026-04-20 19:35:41,411] A new study created in memory with name: no-name-9e2dfb83-c9c9-4540-8b25-9869ab095bc2


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002227 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2305
[LightGBM] [Info] Number of data points in the train set: 36135, number of used features: 29
[LightGBM] [Info] Start training from score 42635.661438
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020498 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2387
[LightGBM] [Info] Number of data points in the train set: 72270, number of used features: 29
[LightGBM] [Info] Start training from score 43730.893433
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.016787 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2411
[LightGBM] [Info] Number of data points in the t

[I 2026-04-20 19:36:00,663] Trial 0 finished with value: 5911.578657104874 and parameters: {'n_estimators': 663, 'num_leaves': 42, 'learning_rate': 0.04957091395476058, 'subsample': 0.6121078554295055, 'colsample_bytree': 0.612862273403987}. Best is trial 0 with value: 5911.578657104874.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002684 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2305
[LightGBM] [Info] Number of data points in the train set: 36135, number of used features: 29
[LightGBM] [Info] Start training from score 42635.661438
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005677 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2387
[LightGBM] [Info] Number of data points in the train set: 72270, number of used features: 29
[LightGBM] [Info] Start training from score 43730.893433
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013925 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is n

[I 2026-04-20 19:36:17,892] Trial 1 finished with value: 5767.128768377807 and parameters: {'n_estimators': 405, 'num_leaves': 126, 'learning_rate': 0.07972829044809913, 'subsample': 0.6148952515731863, 'colsample_bytree': 0.8751099410795926}. Best is trial 1 with value: 5767.128768377807.


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002691 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2305
[LightGBM] [Info] Number of data points in the train set: 36135, number of used features: 29
[LightGBM] [Info] Start training from score 42635.661438
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005310 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2387
[LightGBM] [Info] Number of data points in the train set: 72270, number of used features: 29
[LightGBM] [Info] Start training from score 43730.893433
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008487 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is n

[I 2026-04-20 19:44:44,402] Trial 28 finished with value: 5972.085909378805 and parameters: {'n_estimators': 209, 'num_leaves': 120, 'learning_rate': 0.1473017674867284, 'subsample': 0.8992787828784653, 'colsample_bytree': 0.9410349223728849}. Best is trial 22 with value: 5564.337688439678.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002560 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2305
[LightGBM] [Info] Number of data points in the train set: 36135, number of used features: 29
[LightGBM] [Info] Start training from score 42635.661438
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004843 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2387
[LightGBM] [Info] Number of data points in the train set: 72270, number of used features: 29
[LightGBM] [Info] Start training from score 43730.893433
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007467 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is n

[I 2026-04-20 19:44:59,226] Trial 29 finished with value: 5790.476973701329 and parameters: {'n_estimators': 297, 'num_leaves': 140, 'learning_rate': 0.04673656930322416, 'subsample': 0.8243159293638767, 'colsample_bytree': 0.7452972821669402}. Best is trial 22 with value: 5564.337688439678.


Best Params: {'n_estimators': 294, 'num_leaves': 113, 'learning_rate': 0.014560582047605566, 'subsample': 0.8220865020527265, 'colsample_bytree': 0.9632820171720197}
Best Score: 5564.337688439678


In [102]:
formatted_data = []
for wape, mae, mse, rmse, mape, params in mean_metric_scores:
    # Combine metrics and the params dictionary into one flat dict
    row = {
        'WAPE': wape, 'MAE': mae, 'MSE': mse, 
        'RMSE': rmse, 'MAPE': mape
    }
    row.update(params)
    formatted_data.append(row)

df = pd.DataFrame(formatted_data)
df

,WAPE,MAE,MSE,RMSE,MAPE,n_estimators,num_leaves,learning_rate,subsample,colsample_bytree,random_state
0,0.138148,5911.578657,6.457992e+07,8022.879032,0,663,42,0.049571,0.612108,0.612862,42
1,0.134649,5767.128768,6.240716e+07,7881.649490,0,405,126,0.079728,0.614895,0.875110,42
2,0.147219,6292.973879,7.199119e+07,8478.878547,0,646,134,0.175769,0.809866,0.661866,42
3,0.141567,6052.546914,6.697364e+07,8175.737309,0,444,76,0.133480,0.985123,0.928810,42
4,0.154122,6585.433539,7.782999e+07,8813.504686,0,640,94,0.252334,0.759457,0.899931,42
5,0.145921,6239.074919,7.093730e+07,8417.641415,0,643,65,0.178887,0.604339,0.674126,42
6,0.134058,5742.279253,6.161622e+07,7830.990586,0,494,117,0.029544,0.849402,0.862593,42
7,0.144389,6173.889406,6.984324e+07,8347.682031,0,318,135,0.286164,0.930966,0.855048,42
8,0.141101,6031.296557,6.653895e+07,8151.495527,0,765,49,0.070190,0.875471,0.749882,42
9,0.142572,6097.160664,6.806301e+07,8241.538926,0,386,110,0.184340,0.865129,0.760803,42


In [103]:
df.to_csv('lightgbm_errorfile_sales.csv', header=True, index=False)

In [104]:
study.best_params

{'n_estimators': 294,
 'num_leaves': 113,
 'learning_rate': 0.014560582047605566,
 'subsample': 0.8220865020527265,
 'colsample_bytree': 0.9632820171720197}